# 05 – HDBSCAN Tuning & Modeling (M3)

Phụ trách: **M3 – Thành viên 3 (Hoàng Tuấn Bảo)**. Theo đặc tả BA Spec v1.1 (Phần 2.3.4 & Mục 3.4.3):
- **M3-01 & M3-02**: Khảo sát tham số `min_cluster_size` qua các ngưỡng $[30, 50, 100, 200, 300, 500]$ điểm.
- **M3-03**: Khảo sát tham số `min_samples` qua $[5, 10, 20, 50]$ với `min_cluster_size` cố định để kiểm soát độ bảo thủ mật độ lõi.
- **M3-04**: Chốt cấu hình chính thức và fit mô hình HDBSCAN độc lập trên 3 lát cắt thời gian (`WEEKDAY_AM`, `WEEKDAY_PM`, `WEEKEND`).
- **M3-05**: Đo runtime, mức chiếm dụng RAM, tỷ lệ nhiễu (Noise Ratio), tính các chỉ số nội bộ (Silhouette, Davies-Bouldin, Calinski-Harabasz) trên tập điểm có cụm (`clustered-only`).
- Ghi log vào `outputs/metrics/run_log.csv`, lưu nhãn và xác suất thành viên (`probability`) vào `outputs/labels/<run_id>.parquet`, và xuất bảng chỉ số chính thức `outputs/metrics/hdbscan_metrics.csv`.
- Trực quan hoá bản đồ tương tác Folium HTML đa lớp tại `outputs/maps/hdbscan_map_master_v1.html`.

Dữ liệu sử dụng: **bản đầy đủ v1 do M1 phát hành (`master_v1`)** (4 tuần 17/04–14/05/2023, 535.167 dòng sau làm sạch, 304 toạ độ duy nhất; 3 lát cắt: `WEEKDAY_AM` 60.498, `WEEKDAY_PM` 86.617, `WEEKEND` 120.571 dòng).

In [ ]:
# ===== Ô THIẾT LẬP (chạy được trên cả Google Colab và máy cá nhân) =====
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/tanbentam/KhaiPhaDuLieu.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    ROOT = pathlib.Path("/content/KhaiPhaDuLieu")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
    os.chdir(ROOT)
    from src.colab import link_drive_data
    link_drive_data("KhaiPhaDuLieu")
else:
    here = pathlib.Path.cwd().resolve()
    ROOT = next(p for p in (here, *here.parents) if (p / "config" / "experiment.yaml").is_file())

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Môi trường Colab:", IN_COLAB, "| Thư mục gốc repo:", ROOT)

## 1. Nạp Dữ Liệu & Kiểm Tra Tính Toàn Vẹn

In [ ]:
import hashlib, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.config import load_config, resolve_path

cfg = load_config()
DATA_KEY = "master"  # "master" (master_v1 535k) hoặc "model_input" (sample_v1 100k)
data_path = resolve_path(cfg, DATA_KEY)
man_path = resolve_path(cfg, "master_manifest")
manifest = json.loads(man_path.read_text(encoding="utf-8"))

# Kiểm tra SHA256 để đảm bảo tính tái lập 100% khớp với M1
actual_sha = hashlib.sha256(data_path.read_bytes()).hexdigest()
assert actual_sha == manifest["sha256"], f"Lỗi mã băm: {actual_sha} != {manifest['sha256']}"
print(f"Xác nhận SHA256 thành công: {actual_sha[:16]}...")

df = pd.read_parquet(data_path)
SAMPLE_ID = manifest["sample_id"]
SLICES = list(cfg["time_slices"])
print(f"Nạp thành công {len(df):,} dòng từ {data_path.name} (sample_id: {SAMPLE_ID})")
for s in SLICES:
    print(f"  - Lát cắt {s:11s}: {(df.time_slice == s).sum():,} dòng")

## 2. Khảo Sát Tham Số `min_cluster_size`

Theo BA Spec mục 2.3.4 và cảnh báo R05: trong thư viện `hdbscan`, nếu không truyền `min_samples`, giá trị sẽ tự động phụ thuộc vào `min_cluster_size`. Do đó ta **cố định `min_samples = 10`** và quét `min_cluster_size` qua dải $[30, 50, 100, 200, 300, 500]$ trên lát cắt `WEEKDAY_AM`.

In [ ]:
from src.models.hdbscan import scan_min_cluster_size

am_df = df[df.time_slice == "WEEKDAY_AM"].reset_index(drop=True)
X_am = am_df[["x_m", "y_m"]].to_numpy("float64")

hdb_cfg = cfg.get("hdbscan", {})
mcs_list = hdb_cfg.get("min_cluster_size_candidates", [30, 50, 100, 200, 300, 500])
ms_default = hdb_cfg.get("min_samples_default", 10)

df_mcs = scan_min_cluster_size(
    X_am, mcs_list, min_samples=ms_default,
    metric="euclidean", cluster_selection_method="eom",
    seed=cfg["project"]["seed"], silhouette_max_sample=cfg["evaluation"]["silhouette_max_sample"]
)

mcs_path = resolve_path(cfg, "run_log").parent / f"hdbscan_mcs_tuning_{SAMPLE_ID}.csv"
df_mcs.to_csv(mcs_path, index=False)
print("Đã lưu bảng tuning min_cluster_size:", mcs_path.relative_to(ROOT))
df_mcs[["min_cluster_size", "min_samples", "n_clusters", "noise_ratio", "coverage",
        "size_min", "size_max", "largest_share", "silhouette", "davies_bouldin", "runtime_sec"]]

## 3. Khảo Sát Mức Độ Bảo Thủ Qua Tham Số `min_samples`

Sau khi khảo sát `min_cluster_size`, chọn $mcs = 100$ (cân bằng giữa việc loại bỏ micro-clusters và bảo toàn số vùng hotspot chính), ta tiếp tục quét `min_samples` qua $[5, 10, 20, 50]$.

In [ ]:
from src.models.hdbscan import scan_min_samples

CHOSEN_MCS = 100
ms_candidates = hdb_cfg.get("min_samples_candidates", [5, 10, 20, 50])

df_ms = scan_min_samples(
    X_am, ms_candidates, min_cluster_size=CHOSEN_MCS,
    metric="euclidean", cluster_selection_method="eom",
    seed=cfg["project"]["seed"], silhouette_max_sample=cfg["evaluation"]["silhouette_max_sample"]
)

ms_path = resolve_path(cfg, "run_log").parent / f"hdbscan_ms_tuning_{SAMPLE_ID}.csv"
df_ms.to_csv(ms_path, index=False)
print("Đã lưu bảng tuning min_samples:", ms_path.relative_to(ROOT))
df_ms[["min_cluster_size", "min_samples", "n_clusters", "noise_ratio", "coverage",
       "size_min", "size_max", "largest_share", "silhouette", "davies_bouldin", "runtime_sec"]]

### Trực quan hoá đường cong Tuning HDBSCAN

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

ax1 = axes[0]
ax1.plot(df_mcs["min_cluster_size"], df_mcs["n_clusters"], "o-", color="#1f77b4", label="Số cụm")
ax1.set_xlabel("min_cluster_size (min_samples = 10)")
ax1.set_ylabel("Số cụm (n_clusters)", color="#1f77b4")
ax1.tick_params(axis="y", labelcolor="#1f77b4")
ax1.grid(alpha=0.3)
ax1_twin = ax1.twinx()
ax1_twin.plot(df_mcs["min_cluster_size"], df_mcs["noise_ratio"] * 100, "s--", color="#d62728", label="Noise ratio (%)")
ax1_twin.set_ylabel("Noise ratio (%)", color="#d62728")
ax1_twin.tick_params(axis="y", labelcolor="#d62728")
ax1.set_title("Ảnh hưởng của min_cluster_size (WEEKDAY_AM)")

ax2 = axes[1]
ax2.plot(df_ms["min_samples"], df_ms["noise_ratio"] * 100, "s-", color="#d62728", label="Noise ratio (%)")
ax2.set_xlabel(f"min_samples (min_cluster_size = {CHOSEN_MCS})")
ax2.set_ylabel("Noise ratio (%)", color="#d62728")
ax2.grid(alpha=0.3)
ax2_twin = ax2.twinx()
ax2_twin.plot(df_ms["min_samples"], df_ms["silhouette"], "^-", color="#2ca02c", label="Silhouette Score")
ax2_twin.set_ylabel("Silhouette", color="#2ca02c")
ax2_twin.tick_params(axis="y", labelcolor="#2ca02c")
ax2.set_title("Ảnh hưởng của min_samples (Độ bảo thủ mật độ)")

fig.suptitle(f"HDBSCAN Hyperparameter Tuning - {SAMPLE_ID}", fontsize=13, y=1.02)
fig.tight_layout()
fig_path = resolve_path(cfg, "figures_dir") / f"hdbscan_tuning_{SAMPLE_ID}.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
print("Đã lưu biểu đồ tuning:", fig_path.relative_to(ROOT))

## 4. Chốt Cấu Hình & Fit Chính Thức Trên 3 Lát Cắt

Cấu hình chính thức được lựa chọn: **`min_cluster_size = 100`, `min_samples = 10`, `cluster_selection_method = 'eom'`**.
Cấu hình này đạt tỷ lệ Noise Ratio hợp lý (~10%), tách biệt rõ các trung tâm giao thông chính (Loop, Near North, O'Hare, Midway, Hyde Park) mà không bị gộp/vỡ cụm như DBSCAN.

In [ ]:
from src.evaluation.runlog import run_and_log, export_metrics_table
from src.evaluation.hotspots import relabel_by_size, cluster_table
from src.models.hdbscan import fit_hdbscan

FINAL_MCS = 100
FINAL_MS = 10
FINAL_METHOD = "eom"

final_hdb = {}
final_run_ids = []

for s in SLICES:
    part = df[df.time_slice == s].reset_index(drop=True)
    X = part[["x_m", "y_m"]].to_numpy(dtype="float64")
    params = {
        "min_cluster_size": FINAL_MCS,
        "min_samples": FINAL_MS,
        "cluster_selection_method": FINAL_METHOD,
        "metric": "euclidean",
        "cluster_selection_epsilon": 0.0,
    }

    def _fit_fn(X_arr):
        lab, prob, _ = fit_hdbscan(
            X_arr, min_cluster_size=FINAL_MCS, min_samples=FINAL_MS, cluster_selection_method=FINAL_METHOD
        )
        return relabel_by_size(lab), prob

    row = run_and_log(
        cfg,
        algorithm="hdbscan",
        fit_fn=_fit_fn,
        X=X,
        trip_ids=part["trip_id"].to_numpy(),
        member="M3",
        time_slice=s,
        sample_id=SAMPLE_ID,
        params=params,
        note=f"du lieu M1 {SAMPLE_ID}; cau hinh chinh thuc chot o Giai doan 2",
    )
    final_run_ids.append(row["run_id"])
    lab_df = pd.read_parquet(resolve_path(cfg, "labels_dir") / f"{row['run_id']}.parquet")
    final_hdb[s] = (part, lab_df["label"].to_numpy(), row)

    print(f"{s:11s} -> {row['run_id']} | Cụm: {row['n_clusters']} | Noise: {row['noise_ratio']:.1%} | "
          f"Silhouette: {row['silhouette']:.3f} | DBI: {row['davies_bouldin']:.3f} | Fit: {row['runtime_fit_sec']:.2f}s")

## 5. Xuất Bảng Chỉ Số Đánh Giá & Bảng Thống Kê Cụm

In [ ]:
# Xuất bảng chỉ số hdbscan_metrics.csv chuẩn 28 cột đồng nhất với baseline_metrics.csv của M2
out_metrics_path = resolve_path(cfg, "run_log").parent / "hdbscan_metrics.csv"
metrics_table = export_metrics_table(
    resolve_path(cfg, "run_log"),
    out_metrics_path,
    final_run_ids,
    require_same_sample=True,
)
print("Đã lưu bảng metrics chính thức:", out_metrics_path.relative_to(ROOT))

# Lập bảng chi tiết cụm (hotspot summary table)
cluster_tabs = []
for s, (part, lab, row) in final_hdb.items():
    ct = cluster_table(part, lab, crs=cfg["project"]["crs"])
    ct = ct.assign(
        time_slice=s,
        run_id=row["run_id"],
        sample_id=SAMPLE_ID,
        noise_trips=int((lab == -1).sum()),
        noise_share=float((lab == -1).mean()),
    )
    cluster_tabs.append(ct)

hdb_clusters = pd.concat(cluster_tabs, ignore_index=True)
clp = resolve_path(cfg, "run_log").parent / f"hdbscan_clusters_{SAMPLE_ID}.csv"
hdb_clusters.to_csv(clp, index=False)
print("Đã lưu bảng cụm:", clp.relative_to(ROOT))
hdb_clusters[["time_slice", "cluster", "n_trips", "share", "radius_rms_m", "top_community_area"]]

## 6. Trực Quan Hoá Bản Đồ Tương Tác Folium

Tạo bản đồ Folium HTML đa lớp hiển thị kết quả phân cụm HDBSCAN của cả 3 lát cắt.

In [ ]:
from src.visualization.maps import cluster_map

map_path = resolve_path(cfg, "maps_dir") / f"hdbscan_map_{SAMPLE_ID}.html"
layers = {}
for s, (part, lab, row) in final_hdb.items():
    ct_sub = hdb_clusters[hdb_clusters.time_slice == s]
    layers[f"HDBSCAN {s} (mcs={FINAL_MCS}, ms={FINAL_MS})"] = (part, lab, ct_sub)

cluster_map(
    layers,
    map_path,
    title=f"HDBSCAN - {SAMPLE_ID}",
    note=f"Cấu hình chính thức: min_cluster_size={FINAL_MCS}, min_samples={FINAL_MS}. Dữ liệu v1 ({SAMPLE_ID}).",
)
print("Đã lưu bản đồ Folium:", map_path.relative_to(ROOT), f"({map_path.stat().st_size / 1024:.0f} KB)")